In [ ]:
# ========== 导入：后面抓网页、读密钥、调 Gemini 都要用到这些工具 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 GOOGLE_API_KEY
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进进程环境，避免把密钥写进代码
from dotenv import load_dotenv
# 从同目录 scraper 模块导入 fetch_website_contents：抓取网页正文（本格先导入备用）
from scraper import fetch_website_contents
# 从 IPython.display 导入 Markdown、display：在笔记本里漂亮地渲染 Markdown
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：后面用 OpenAI 兼容协议调用 Google Gemini
from openai import OpenAI


## 加载并验证 Google API 密钥

用 `load_dotenv()` 从 `.env` 读入 `GOOGLE_API_KEY`，再做几项健全性检查（sanity checks）：

| 检查项 | 含义 |
|--------|------|
| **缺少密钥** | 环境里找不到 key → 打印警告，指引去排查笔记本 |
| **格式不对** | Google API 密钥通常以 `AI` 开头；若不匹配则提示可能拿错了 key |
| **首尾空白** | 捕获密钥前后多余空格/制表符（复制粘贴常见坑） |
| **看起来正常** | 通过上述检查后，确认可以继续往下跑 |

> 注意：下面代码里 `startswith("AI")` 失败时的英文提示文案仍写着 `sk-proj-`（OpenAI 风格）。这是原作者笔误，**不要改打印字符串**——它不影响 `startswith` 判断本身，只是文案不准确。


In [ ]:
# ========== 读密钥 + 健全性检查：确认 GOOGLE_API_KEY 可用再往下调 API ==========

# 加载 .env；override=True 表示用文件里的值覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 从环境变量取出 Google API Key（没有则为 None）
api_key = os.getenv('GOOGLE_API_KEY')

# —— 按优先级做几项检查；错误提示字符串保持英文原样（排查笔记本指引依赖这些文案）——

# 情况 1：完全没读到密钥
if not api_key:
    # 提示去同目录 troubleshooting 笔记本排查（文案保留英文）
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
# 情况 2：有密钥，但不是 Google 常见的 AI... 前缀
elif not api_key.startswith("AI"):
    # 原作者提示文案写成了 sk-proj-（OpenAI 风格）；逻辑仍按 startswith("AI") 判断，字符串勿改
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
# 情况 3：首尾有空白（strip 后与原文不同）
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
# 情况 4：通过检查
else:
    print("API key found and looks good so far!")


## 列出可用的 Google Gemini 模型

用官方 `google-genai` 客户端，拿刚才读到的 API 密钥初始化连接，再检索当前密钥能访问的全部模型。

- `genai.Client(api_key=...)`：创建客户端
- `client.models.list()`：返回可访问模型的可迭代列表
- 循环打印每个模型的 `name`，方便后面选 `gemini-2.5-flash` 等

这和本课 Day 1「先确认服务端通了，再发第一条消息」是同一思路。


In [ ]:
# ========== 用 google-genai 官方 SDK 列出当前密钥可见的模型 ==========

# 从 google 包导入 genai：Google 新一代 Generative AI 官方客户端
from google import genai

# 用前面校验过的 api_key 创建 Client（密钥走构造参数，不写死在源码里）
client = genai.Client(api_key=api_key)

# 拉取模型列表（Pager / 可迭代对象，具体类型随 SDK 版本可能略有差异）
models = client.models.list()

# 逐个打印模型名，例如 models/gemini-2.5-flash
for m in models:
    print(m.name)


In [ ]:
# ========== 预览 messages：Chat Completions 的「对话列表」长什么样 ==========
# 用这些 messages 调用（兼容）OpenAI 接口就是这么简单；有问题请看同目录 troubleshooting 笔记本。

# 用户要发给模型的一句英文招呼（可运行字符串，保持英文）
message = "Hello, GPT! This is my first ever message to you! Hi!"

# 组装 messages：一条 user 消息；role/content 是 Chat Completions 标准字段
messages = [{"role": "user", "content": message}]

# 在 Jupyter 里单独写变量名：作为单元格最后表达式，会显示 messages 的内容供你检查
messages


In [ ]:
# ========== 经 OpenAI 兼容端点调用 Gemini：同一套 chat.completions API ==========

# Google Generative Language 的 OpenAI 兼容 base URL（路径 /v1beta/openai 勿改）
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai"

# 创建 OpenAI 客户端，但 base_url 指到 Google；api_key 仍用 Google 的密钥
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)
# 发起一次非流式 Chat Completions；model id 必须是 Gemini 侧可用的名字
response = gemini.chat.completions.create(model="gemini-2.5-flash", messages= messages)
# 取出第一条 choice 里 assistant 的文本内容并打印
print(response.choices[0].message.content)
